# SC Data tutorial

Basic python tools to explore the data of a few devices: an enclosure test with four kits (February 2025).

```
import scdata as sc
```

## Load the packages

In [ ]:
import logging

import scdata as sc
import smartcitizen_connector
from scdata.tools.custom_logger import set_logger_level

# Only errors: 'WARNING', 'INFO' or 'DEBUG' show what scdata and smartcitizen-connector do
set_logger_level('ERROR')
logging.getLogger('smartcitizen_connector').setLevel('ERROR')

sc.__version__

## A device

A device without data yet:

In [ ]:
d = sc.Device(blueprint='sc_air', params=sc.APIParams(id=18486))
d.data

Load the days of the test (without dates, it loads all the data of the device):

In [ ]:
d.options.min_date = '2025-02-12'
d.options.max_date = '2025-03-07'
await d.load()

In [ ]:
d.data

In [ ]:
type(d.data)

A `pandas.DataFrame`: all pandas works on it.

In [ ]:
d.data.describe()['HUM']

In [ ]:
d.data[['TEMP', 'HUM', 'NOISE_A']].agg(['mean', 'max', 'min'])

In [ ]:
# Needs pip install "scdata[plot]" (matplotlib)
d.data['NOISE_A'].plot(figsize=(20, 6));

## A test: the four kits together

In [ ]:
devices_ids = [18486, 18487, 18488, 18489]

devices = []
for device_id in devices_ids:
    device = sc.Device(blueprint='sc_air', params=sc.APIParams(id=device_id))
    device.options.min_date = '2025-02-12'
    device.options.max_date = '2025-03-07'
    devices.append(device)

t = sc.Test(name='SCK2.3_ENCLOSURE_TEST', devices=devices, force_recreate=True)
await t.load()

In [ ]:
t.get_device(18486).data.columns

## Plots

In [ ]:
traces = {
    '1': {'devices': 'all', 'channel': 'TEMP', 'subplot': 1},
    '2': {'devices': 'all', 'channel': 'HUM', 'subplot': 2},
}

t.ts_uplot(traces=traces,
           formatting={'width': 1200, 'height': 400, 'padding-bottom': 600, 'title': 'Temperature and humidity',
                       'ylabel': {1: 'Temperature (degC)', 2: 'Humidity (%rh)'}},
           options={'frequency': '30min'})

## Selecting periods

Rows of a period, by date:

In [ ]:
df = t.get_device(18486).data
df.loc['2025-02-12 14:07:01+01:00':'2025-02-13 14:07:01+01:00']

In [ ]:
import pandas as pd

# Mean of each device over the whole test
pd.DataFrame({device.id: device.data[['TEMP', 'HUM']].mean() for device in t.devices}).round(2)

## First day

Two kits with the old enclosure (slots) and two with the new one: the UVA light inside each one.

In [ ]:
slots = [18486, 18487]
new = [18488, 18489]

date_min = '2025-02-19 14:55:00+01:00'
date_max = '2025-02-20 15:00:01+01:00'

for device_id in devices_ids:
    device_type = 'slots' if device_id in slots else 'new'
    device = t.get_device(device_id)
    print(device_type, device_id, '----', device.data.loc[date_min:date_max, 'AS7331_UVA'].mean())

In [ ]:
t.ts_uplot(traces={'1': {'devices': 'all', 'channel': 'AS7331_UVA', 'subplot': 1}},
           formatting={'width': 1200, 'height': 400, 'title': 'UVA', 'ylabel': {1: 'UVA (uW/cm2)'}},
           options={'frequency': '30min', 'min_date': date_min, 'max_date': date_max})

## Second period

In [ ]:
date_min = '2025-02-20 15:30:00+01:00'
date_max = '2025-03-06 11:30:01+01:00'

for device_id in devices_ids:
    device_type = 'slots' if device_id in slots else 'new'
    print(device_type, device_id, '----', t.get_device(device_id).data.loc[date_min:date_max, 'AS7331_UVA'].mean())

In [ ]:
t.ts_uplot(traces=traces,
           formatting={'width': 1200, 'height': 400, 'padding-bottom': 600, 'title': 'Temperature and humidity, second period',
                       'ylabel': {1: 'Temperature (degC)', 2: 'Humidity (%rh)'}},
           options={'frequency': '30min', 'min_date': date_min, 'max_date': date_max})